# Track T3: XGBoost Modeling, Hyperparameter Tuning & Tree-SHAP Explainability
**Student Owner:** B1 (Feature Specialist & Gradient Boosting Specialist)

This notebook covers the complete Track T3 machine learning workflow for the Solar Inverter Health Monitoring System:
1. **Stage 9 Supervised Modeling:** Continuous `xgb_regressor` and 3-class `xgb_classifier`, both forecasting **Health(t + 1 h)** from features at t (Option A, section 7.2). An earlier version predicted Health(t) from PR features at t, which rebuilt the target from its own inputs (target leakage).
2. **Architectural Comparison:** Method 1 (Dedicated Softmax Classifier) vs. Method 2 (Thresholded Regressor).
3. **Stage 10 Hyperparameter Tuning:** Grid sweep using **B2's defined chronological splits** (`train_df` and `val_df`) (validation RMSE, no early stopping).
4. **Stage 5 Balancing:** sample weights vs Healthy undersampling on the validation set.
5. **Stage 11 Evaluation:** Generalization across the full 2023 chronological test set and 7 Unseen Stations.
6. **Default vs. Tuned Comparison Table.**
7. **Stage 11.3 Tree-SHAP Explainability:** Global feature importance beeswarm, local failure waterfall plot, and the observed context of the top 3 failure cases.

In [1]:
import sys
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features import STAGE_7_SELECTED_FEATURES, build_all_features
from src.evaluate import split_chronological, split_unseen_stations, evaluate_regression, evaluate_classification, log_experiment
from src.models.xgb import (
    build_xgb_regressor,
    build_xgb_classifier,
    build_forecast_dataset,
    make_splits,
    train_fill_values,
    prepare_xgb_data,
    compare_balancing,
    undersample_healthy,
    compare_classification_strategies,
    run_hyperparameter_tuning,
    run_shap_explainability,
    load_or_generate_dataset,
    threshold_health_score,
    compute_sample_weights,
    TUNED_REG_PARAMS,
    DEFAULT_CLS_PARAMS
)
print("Setup completed successfully. All dependencies and B1/B2 modules imported.")

Setup completed successfully. All dependencies and B1/B2 modules imported.


## 1. Load Telemetry & Apply B2's Chronological Splitting Engine

To strictly eliminate data leakage, we adopt **B2's defined Chronological Splitting Engine** (`split_chronological` and `split_unseen_stations`):
- **Train Set:** Data up to June 30, 2022 (18 months)
- **Validation Set:** July 1, 2022 to December 31, 2022 (6 months dedicated tuning period)
- **Test Set:** Full year 2023 (12 months, all 4 seasons)
- **Unseen Stations Set:** 7 completely held-out rooftop stations in 2023

In [2]:
# Load processed dataset (or realistic multi-station sample)
raw_dataset = load_or_generate_dataset()
print(f"Loaded dataset: {len(raw_dataset):,} rows across {raw_dataset['inverter_id'].nunique()} inverters.")

# 1. Forecasting dataset: Stage 7 features at t (built on the full grid), target Health(t + 1 h)
forecast = build_forecast_dataset(raw_dataset)

# 2. B2's split functions; held-out inverters stay out of train and val
splits = make_splits(forecast)
train_df, val_df, test_df, test_unseen = (splits[k] for k in ["train", "val", "test", "test_unseen"])

# 3. Feature matrices (13 Stage 7 features + Health(t)), filled with training medians only
fill_values = train_fill_values(train_df)
X_train, y_reg_train, y_cls_train, _ = prepare_xgb_data(train_df, fill_values)
X_val, y_reg_val, y_cls_val, _ = prepare_xgb_data(val_df, fill_values)
X_test, y_reg_test, y_cls_test, test_clean = prepare_xgb_data(test_df, fill_values)
X_unseen, y_reg_unseen, y_cls_unseen, _ = prepare_xgb_data(test_unseen, fill_values)

print("\n--- Feature Matrices Prepared ---")
print(f"Train  : {X_train.shape[0]:,} rows x {X_train.shape[1]} features")
print(f"Val    : {X_val.shape[0]:,} rows x {X_val.shape[1]} features")
print(f"Test   : {X_test.shape[0]:,} rows (All 2023)")
print(f"Unseen : {X_unseen.shape[0]:,} rows (7 unseen stations)")


Loading real telemetry from data\processed\hkust_15min_v1.parquet...


Loaded dataset: 1,588,901 rows across 44 inverters.


Chronological Split: 261,536 train, 183,667 val, 369,090 test


Unseen Station Split: 261,536 train (known), 37,060 test (unseen)

--- Feature Matrices Prepared ---
Train  : 261,536 rows x 14 features
Val    : 183,667 rows x 14 features
Test   : 369,090 rows (All 2023)
Unseen : 37,060 rows (7 unseen stations)


## 2. Stage 10: Hyperparameter Tuning on B2's Chronological Sets

Rather than using generic mathematical k-fold slicing, candidate configurations are evaluated directly across **B2's defined chronological sets** (`train_df` and `val_df`) by validation RMSE (no early stopping).

In [3]:
# Execute tuning sweep on B2's train and validation sets
best_params, tuning_summary = run_hyperparameter_tuning(train_df, val_df, sample_fraction=0.20, fill_values=fill_values)
tuning_summary


STAGE 10: HYPERPARAMETER TUNING ON B2 CHRONOLOGICAL SPLITS
Tuning Search Space: 52,307 time-ordered train rows -> 183,667 B2 validation rows.


Config 01 | depth=4, lr=0.05, sub=0.8 | Val RMSE: 13.243 | Train RMSE: 7.614 (2.7s)


Config 02 | depth=6, lr=0.03, sub=0.85 | Val RMSE: 14.288 | Train RMSE: 7.101 (3.7s)


Config 03 | depth=6, lr=0.05, sub=0.8 | Val RMSE: 14.579 | Train RMSE: 6.628 (4.0s)


Config 04 | depth=8, lr=0.02, sub=0.8 | Val RMSE: 14.808 | Train RMSE: 6.607 (4.7s)


Config 05 | depth=8, lr=0.05, sub=0.7 | Val RMSE: 14.744 | Train RMSE: 5.734 (5.0s)

Saved Stage 10 tuning summary table: reports\tables\b1_xgb_default_vs_tuned.csv
Optimal Configuration Selected: max_depth=4, lr=0.05 (Val RMSE: 13.243)


,config_id,max_depth,learning_rate,subsample,colsample_bytree,reg_alpha,reg_lambda,train_rmse,val_rmse,val_mae,runtime_s
0,1,4,0.05,0.80,0.80,0.01,1.0,7.614,13.243,6.491,2.7
1,2,6,0.03,0.85,0.85,0.10,1.5,7.101,14.288,6.864,3.7
2,3,6,0.05,0.80,0.80,0.05,1.0,6.628,14.579,6.970,4.0
3,5,8,0.05,0.70,0.70,0.50,2.5,5.734,14.744,7.152,5.0
4,4,8,0.02,0.80,0.75,0.20,2.0,6.607,14.808,7.208,4.7


## 3. Fit Final Tuned Models & Method 1 vs. Method 2 Comparison

We train:
1. **Tuned `xgb_regressor`** predicting continuous Health Score one hour ahead (0 to 100).
2. **Dedicated `xgb_classifier` (Method 1)**, balanced with the Stage 5 winner (sample weights vs Healthy undersampling, chosen on validation Faulty recall then macro-F1).
3. Compare Method 1 against **Method 2 (Thresholded Regressor at 70/90)** on B2's validation fold.

In [4]:
# 1. Fit Tuned Regressor
xgb_reg = build_xgb_regressor(best_params)
t0 = time.time()
xgb_reg.fit(X_train, y_reg_train, verbose=False)
print(f"Fitted tuned xgb_regressor in {time.time() - t0:.2f}s")

# 2. Stage 5 balancing: sample weights vs Healthy undersampling (validation)
balancing_table, balancing = compare_balancing(X_train, y_cls_train, X_val, y_cls_val)

# 3. Fit Dedicated Classifier (Method 1) with the winning strategy
if balancing == "sample_weights":
    X_cls_fit, y_cls_fit, weights = X_train, y_cls_train, compute_sample_weights(y_cls_train)
else:
    (X_cls_fit, y_cls_fit), weights = undersample_healthy(X_train, y_cls_train), None
xgb_cls = build_xgb_classifier(DEFAULT_CLS_PARAMS)
t1 = time.time()
xgb_cls.fit(X_cls_fit, y_cls_fit, sample_weight=weights, verbose=False)
print(f"Fitted dedicated xgb_classifier ({balancing}) in {time.time() - t1:.2f}s")

# 4. Execute Method 1 vs. Method 2 Comparison
comp_results = compare_classification_strategies(xgb_reg, xgb_cls, X_val, y_cls_val)


Fitted tuned xgb_regressor in 3.71s



Stage 5 balancing (validation):
      strategy  train_rows  val_rows  macro_f1  faulty_recall  faulty_precision  faulty_pr_auc  selected
sample_weights      261536    183667    0.5999         0.6542            0.5025         0.6742      True
 undersampling      116943    183667    0.6429         0.5164            0.7603         0.6940     False


Fitted dedicated xgb_classifier (sample_weights) in 11.64s



CLASSIFICATION STRATEGY COMPARISON (ON B2 VALIDATION SET)
Method 1 (Dedicated Softmax Classifier):
  Macro-F1       : 0.5999
  Faulty Recall  : 0.6542 (Primary Metric)
  Faulty Precision: 0.5025

Method 2 (Thresholded Continuous Regressor):
  Macro-F1       : 0.6424
  Faulty Recall  : 0.4824 (Primary Metric)
  Faulty Precision: 0.8865

Recommended Winner: Method 1 (Dedicated Classifier)


## 4. Stage 11: Final Evaluation via B2 Framework (Chronological & Unseen Stations)

We compute MAE, RMSE, R2, Macro-F1, and Faulty Recall across all chronological splits and the 7 unseen rooftop stations.

In [5]:
# Regression Metrics
reg_train = evaluate_regression(y_reg_train, xgb_reg.predict(X_train))
reg_val = evaluate_regression(y_reg_val, xgb_reg.predict(X_val))
reg_test = evaluate_regression(y_reg_test, xgb_reg.predict(X_test))
reg_unseen = evaluate_regression(y_reg_unseen, xgb_reg.predict(X_unseen))

# Classification Metrics (Method 1)
cls_train = evaluate_classification(y_cls_train, xgb_cls.predict(X_train))
cls_val = evaluate_classification(y_cls_val, xgb_cls.predict(X_val))
cls_test = evaluate_classification(y_cls_test, xgb_cls.predict(X_test))
cls_unseen = evaluate_classification(y_cls_unseen, xgb_cls.predict(X_unseen))

summary_data = {
    "Split": ["Train (<= Jun 2022)", "Val (Jul-Dec 2022)", "Test (2023 Chronological)", "Unseen Stations (2023)"],
    "Reg RMSE": [reg_train["rmse"], reg_val["rmse"], reg_test["rmse"], reg_unseen["rmse"]],
    "Reg MAE": [reg_train["mae"], reg_val["mae"], reg_test["mae"], reg_unseen["mae"]],
    "Reg R2": [reg_train["r2"], reg_val["r2"], reg_test["r2"], reg_unseen["r2"]],
    "Cls Macro-F1": [cls_train["macro_f1"], cls_val["macro_f1"], cls_test["macro_f1"], cls_unseen["macro_f1"]],
    "Faulty Recall": [cls_train["faulty_recall"], cls_val["faulty_recall"], cls_test["faulty_recall"], cls_unseen["faulty_recall"]]
}
summary_data["Persistence RMSE"] = [evaluate_regression(p["health_lead"], p["health_now"])["rmse"]
                                    for p in [train_df, val_df, test_df, test_unseen]]
summary_df = pd.DataFrame(summary_data)
summary_df

,Split,Reg RMSE,Reg MAE,Reg R2,Cls Macro-F1,Faulty Recall,Persistence RMSE
0,Train (<= Jun 2022),8.688,4.419,0.625,0.6207,0.7682,12.109
1,Val (Jul-Dec 2022),11.378,6.096,0.600,0.5999,0.6542,13.041
2,Test (2023 Chronological),10.582,5.648,0.555,0.5641,0.6406,12.304
3,Unseen Stations (2023),9.902,4.755,0.252,0.4139,0.4204,10.728


## 5. Stage 11.3: Tree-SHAP Explainability & Physical Failure Diagnosis

We compute exact Shapley attributions using `shap.TreeExplainer` on the 2023 test set:
1. **Global Beeswarm Summary:** Feature importance ranking.
2. **Local Waterfall Attribution:** Severe prediction failure case.
3. **Failure cases:** observed context (Health(t), irradiance, temperature, data validity) of the top 3 errors. Causes are hypotheses; the dataset has no maintenance logs.

In [6]:
# Execute Tree-SHAP and failure case analysis
failure_cases = run_shap_explainability(xgb_reg, X_test, test_clean, sample_size=1500)
failure_cases


STAGE 11.3: TREE-SHAP EXPLAINABILITY & FAILURE CASE DIAGNOSIS
Computing Tree-SHAP values across 1,500 test intervals...


C:\Users\Veer\Documents\SEM_5\ML\ML_Assignment\InvertML\src\models\xgb.py:434: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, X_shap, show=False)


Saved: reports\figures\b1_shap_summary.png


Saved: reports\figures\b1_shap_waterfall.png
Saved failure cases report: reports\tables\b1_xgb_failure_cases.csv

Top 3 Worst Prediction Failures:
  #1 | Inverter: zone_a6_inverter | Health(t): 100.0 | Actual t+1h: 33.8 vs Pred: 97.93 (Err: 64.13)
  #2 | Inverter: zone_a4_inverter | Health(t): 100.0 | Actual t+1h: 34.81 vs Pred: 97.72 (Err: 62.9)
  #3 | Inverter: indoor_sports_centre_inverter | Health(t): 10.9 | Actual t+1h: 7.05 vs Pred: 69.09 (Err: 62.04)


,rank,timestamp,inverter_id,health_now,actual_health_t_plus_1h,predicted_health,residual_error,irradiance_wm2,temp_c,n_valid_5min,power_missing
0,1,2023-08-30 14:00:00,zone_a6_inverter,100.0,33.80,97.93,64.13,589.6,35.8,3,False
1,2,2023-07-20 14:15:00,zone_a4_inverter,100.0,34.81,97.72,62.90,542.4,30.7,3,False
2,3,2023-01-17 11:00:00,indoor_sports_centre_inverter,10.9,7.05,69.09,62.04,295.6,18.4,3,False


## 6. Official Benchmark Logging to `results/results.csv`

Log official scores using B2's `log_experiment` to benchmark XGBoost against B2's MLP and A1's baselines.

In [7]:
LOG_RESULTS = False   # set True once for the final run; each run appends rows to results.csv

if LOG_RESULTS:
  log_experiment(
    student="B1",
    model_name="XGBRegressor (Tuned)",
    task="regression",
    train_score=f"RMSE: {reg_train['rmse']:.2f}",
    val_score=f"RMSE: {reg_val['rmse']:.2f}",
    test_score=f"RMSE: {reg_test['rmse']:.2f}",
    unseen_score=f"RMSE: {reg_unseen['rmse']:.2f}",
    notes=f"Health(t+1h) forecast; depth={best_params['max_depth']}, lr={best_params['learning_rate']}, Stage 7 features + Health(t)"
  )
  log_experiment(
    student="B1",
    model_name="XGBClassifier",
    task="classification",
    train_score=f"F1: {cls_train['macro_f1']:.2f}, Faulty recall: {cls_train['faulty_recall']:.2f}",
    val_score=f"F1: {cls_val['macro_f1']:.2f}, Faulty recall: {cls_val['faulty_recall']:.2f}",
    test_score=f"F1: {cls_test['macro_f1']:.2f}, Faulty recall: {cls_test['faulty_recall']:.2f}",
    unseen_score=f"F1: {cls_unseen['macro_f1']:.2f}, Faulty recall: {cls_unseen['faulty_recall']:.2f}",
    notes=f"Status(t+1h) forecast; dedicated 3-class softmax, {balancing} (Stage 5 winner)"
  )
  print("Official benchmark results logged successfully to results/results.csv.")
else:
  print("LOG_RESULTS is False: nothing written to results/results.csv.")

LOG_RESULTS is False: nothing written to results/results.csv.
